# <u>MODELADO:</u>

In [38]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [39]:
df = pd.read_csv('../data/processed/model_cars.csv')

## <u>1. Separación del dataset</u>

In [40]:
X = df.drop(columns=['price'])
y = df['price']

##### Train + temp (validación+test)

In [41]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42
)

##### Validación y test a partir de temp

In [42]:
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print(X_train.shape, X_val.shape, X_test.shape)

(354658, 26) (118220, 26) (118220, 26)


##### Función de entrenamiento y evaluación del modelo

Para evitar duplicar código y facilitar la comparación entre distintos modelos,
se define una función que encapsula todo el proceso de:

- Preprocesado de variables numéricas y categóricas
- Entrenamiento del modelo de regresión
- Evaluación sobre el conjunto de validación

La función recibe como argumento la lista de variables numéricas, lo que permite
comparar fácilmente diferentes representaciones de una misma variable (por ejemplo,
`mileage` frente a `log_mileage`) manteniendo constante el resto del pipeline.


In [43]:
def train_and_evaluate(num_features, X_train, y_train, X_val, y_val):
    
    num_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    cat_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(
            handle_unknown='ignore',
            min_frequency=0.01
        ))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', num_transformer, num_features),
            ('cat', cat_transformer, cat_features)
        ]
    )

    model = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', LinearRegression())
    ])

    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)

    mae = mean_absolute_error(y_val, y_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))

    return mae, rmse

##### MODELO A(con mileage)

In [44]:
num_features_mileage = [
    'car_age',
    'mileage',
    'mileage_per_year',
    'engine_liters',
    'engine_cylinders',
    'engine_power_index',
    'mpg_mean',
    'driver_reviews_num'
]

mae_mileage, rmse_mileage = train_and_evaluate(
    num_features_mileage,
    X_train, y_train,
    X_val, y_val
)

print("Modelo con mileage")
print(f"MAE: {mae_mileage:.2f}")
print(f"RMSE: {rmse_mileage:.2f}")

Modelo con mileage
MAE: 6398.01
RMSE: 10394.73


##### MODELO B (con log_mileage)

In [45]:
num_features_log = [
    'car_age',
    'log_mileage',
    'mileage_per_year',
    'engine_liters',
    'engine_cylinders',
    'engine_power_index',
    'mpg_mean',
    'driver_reviews_num'
]

mae_log, rmse_log = train_and_evaluate(
    num_features_log,
    X_train, y_train,
    X_val, y_val
)

print("Modelo con log_mileage")
print(f"MAE: {mae_log:.2f}")
print(f"RMSE: {rmse_log:.2f}")

Modelo con log_mileage
MAE: 6427.72
RMSE: 10439.70


##### Comparación final

In [46]:
results = pd.DataFrame({
    'Modelo': ['Mileage', 'Log mileage'],
    'MAE': [mae_mileage, mae_log],
    'RMSE': [rmse_mileage, rmse_log]
})

results

,Modelo,MAE,RMSE
0,Mileage,6398.009065,10394.730766
1,Log mileage,6427.718166,10439.700861


##### Conclusión de la comparación mileage vs log(mileage)

Se entrenaron dos modelos idénticos, variando únicamente la forma de representar
el kilometraje del vehículo.

Los resultados muestran que el uso del kilometraje original (`mileage`) obtiene
ligeramente mejores valores de MAE y RMSE que la transformación logarítmica
(`log_mileage`).

Por tanto, en este caso, la transformación logarítmica no mejora el rendimiento
predictivo del modelo y se opta por mantener la variable `mileage` en su forma
original para los siguientes experimentos.


##### Evaluar el mejor modelo (Mileage) en el conjunto de test

In [48]:
best_num_features = num_features_mileage  # el que mejor ha ido

mae_test, rmse_test = train_and_evaluate(
    best_num_features,
    X_train, y_train,
    X_test, y_test
)

print(f"MAE test: {mae_test:.2f}")
print(f"RMSE test: {rmse_test:.2f}")

MAE test: 6413.75
RMSE test: 10428.28
